# tensor_101：v0.py 里的 tensor

[kernels/gemm/v0.py](../kernels/gemm/v0.py) 用 tensor 只做了四件事：

```python
# 1. 指针 + 新 layout = 新 tensor
A = fx.make_view(fx.get_iter(A), fx.make_ordered_layout((8, K//8, 16, BM//16, grid_m), 0))
# 2. 用 None 切片，选出本 CTA 负责的块
A = A[None, None, None, None, bid_m]
# 3. 每个 lane 读 8 个连续 bf16
a = A[None, kk, row, i].load()
# 4. 逐个元素写回 C
C[row, j, kg * 4 + r, i] = out[r]
```

下面用最小的 kernel 把这几行逐个拆开，看它们在 trace 时得到什么类型，最后变成什么 ISA。

- ISA 用 `ARCH=gfx942 COMPILE_ONLY=1` 编译得到，本机 GPU 不必是 gfx942；
- 数值正确性在本机 GPU 上跑，和 torch 的结果对比。

## 0. 准备

几个小工具：

- `make_launcher(kernel_fn, grid)`：把 `kernel_fn(A, O)` 包成 64 lane/CTA 的 launcher；
- `run(...)`：在本机 GPU 上运行；
- `isa(launcher, *args)`：编译成 gfx942 汇编，返回 `.s` 全文；kernel 里 `print` 的内容照常输出；
- `body(text)`：从 `.s` 里取出 kernel 主体指令（去掉注释和伪指令）；
- `trace(kernel_fn, A, O)`：只编译，用来看 trace 时 `print` 出来的类型。

In [1]:
import os
os.environ["FLYDSL_RUNTIME_ENABLE_CACHE"] = "0"  # 不用磁盘缓存，保证每次都重新 trace

import contextlib, glob, io, re, tempfile
from importlib.metadata import version

import torch
import flydsl.compiler as flyc
import flydsl.expr as fx

ISA_ARCH = "gfx942"
stream = fx.Stream(torch.cuda.current_stream().cuda_stream)
print("flydsl", version("flydsl"), "| torch", torch.__version__,
      "| GPU", torch.cuda.get_device_properties(0).gcnArchName)


def make_launcher(kernel_fn, grid=(1, 1, 1)):
    # 每次新建 kernel/launcher：编译 gfx942 ISA 和在本机 GPU 运行时不共用同一个 JitFunction
    kernel = flyc.kernel(known_block_size=(64, 1, 1))(kernel_fn)

    @flyc.jit
    def launch(A: fx.Tensor, O: fx.Tensor, stream: fx.Stream):
        kernel(A, O).launch(grid=grid, block=(64, 1, 1), stream=stream)

    return launch


def run(kernel_fn, A, O, grid=(1, 1, 1)):
    make_launcher(kernel_fn, grid)(A, O, stream)
    torch.cuda.synchronize()


def isa(launcher, *args):
    dump_dir = tempfile.mkdtemp(prefix="tensor101_")
    env = dict(ARCH=ISA_ARCH, COMPILE_ONLY="1", FLYDSL_DUMP_IR="1", FLYDSL_DUMP_DIR=dump_dir)
    old = {k: os.environ.get(k) for k in env}
    os.environ.update(env)
    out = io.StringIO()
    try:
        with contextlib.redirect_stdout(out):
            flyc.compile(launcher, *args)
    finally:
        for k, v in old.items():
            if v is None:
                os.environ.pop(k)
            else:
                os.environ[k] = v
    for line in out.getvalue().splitlines():
        if not line.startswith("[flydsl.compile]"):
            print(line)
    path, = glob.glob(f"{dump_dir}/**/*final_isa.s", recursive=True)
    return open(path).read()


def body(text):
    lines, inside = [], False
    for raw in text.splitlines():
        if re.match(r"^\w+:", raw):          # kernel 入口标签
            inside = True
            continue
        line = raw.split(";")[0].strip()
        if inside and line and (not line.startswith(".") or line.startswith(".LBB")):
            lines.append(line)
        if line.startswith("s_endpgm"):
            break
    return lines


def trace(kernel_fn, A, O):
    isa(make_launcher(kernel_fn), A, O, stream)

flydsl 0.3.4.1 | torch 2.9.1+rocm7.2.0.git7e1940d4 | GPU gfx950:sramecc+:xnack-


## 1. kernel 参数里的 `fx.Tensor`

先看 torch 张量传进 kernel 之后变成了什么。

In [2]:
A = torch.randn(32, 64, dtype=torch.bfloat16, device="cuda")
O = torch.zeros(64, 8, dtype=torch.bfloat16, device="cuda")


def show_arg(A: fx.Tensor, O: fx.Tensor):
    print("  A        :", A.type)
    print("  A.layout :", A.layout.type)
    print("  iter     :", fx.get_iter(A).type)


for name, t in [("A", A), ("A.T", A.T), ("A[0]", A[0]), ("A[:, ::2]", A[:, ::2])]:
    print(f"{name}: torch shape={tuple(t.shape)} stride={t.stride()}")
    try:
        trace(show_arg, t, O)
    except Exception as e:
        print("  error:", str(e).strip().splitlines()[0][:150])

A: torch shape=(32, 64) stride=(64, 1)
  A        : !fly.memref<bf16, global, (?,?):(?{i64},1)>
  A.layout : !fly.layout<(?,?):(?{i64},1)>
  iter     : !fly.ptr<bf16, global>
A.T: torch shape=(64, 32) stride=(1, 64)
  A        : !fly.memref<bf16, global, (?,?):(1,?{i64})>
  A.layout : !fly.layout<(?,?):(1,?{i64})>
  iter     : !fly.ptr<bf16, global>
A[0]: torch shape=(64,) stride=(1,)
  A        : !fly.memref<bf16, global, ?:1>
  A.layout : !fly.layout<?:1>
  iter     : !fly.ptr<bf16, global>
A[:, ::2]: torch shape=(32, 32) stride=(64, 2)
  error: cannot auto-mark layout-dynamic for tensor shape=(32, 32) strides=(64, 2): tensor has no axis with stride == 1; layout-dynamic memref requires one. Us


- kernel 里的 `fx.Tensor` 是 `!fly.memref<dtype, 地址空间, shape:stride>`，也就是**一个指针加一个 layout**。
- torch 张量传进来时，shape 和 stride 都是 `?`（运行时才知道的值，stride 是 i64）；只有 stride 为 1 的那一维被标成静态的 `1`。转置后的 `A.T` 就是 `(?,?):(1,?)`。
- 没有 stride=1 维的张量（`A[:, ::2]`）不能这样直接传进来，报错信息提示改用 `flyc.from_dlpack`。
- `fx.get_iter(A)` 只取出指针 `!fly.ptr<bf16, global>`，丢掉 layout。

## 2. layout：shape:stride 与 `make_ordered_layout`

layout 把坐标 $(c_0, c_1, \dots)$ 映射成偏移 $\sum_i c_i \cdot s_i$（单位是元素），和 torch 的 shape/stride 是一回事。

`make_ordered_layout(shape, order)` 只给 shape 和各维的快慢顺序，stride 按紧凑排列自动算出来；`make_layout(shape, stride)` 则直接给 stride。

In [3]:
def show_layouts(A: fx.Tensor, O: fx.Tensor):
    shape = (8, 8, 16, 2)
    for order in [0, (0, 1, 2, 3), (3, 2, 1, 0), (1, 0, 3, 2)]:
        print(f"make_ordered_layout(shape, {str(order):12}) ->", fx.make_ordered_layout(shape, order).type)
    print(f"make_layout(shape, (1, 8, 64, 1024))      ->", fx.make_layout(shape, (1, 8, 64, 1024)).type)


trace(show_layouts, A, O)

make_ordered_layout(shape, 0           ) -> !fly.layout<(8,8,16,2):(1,8,64,1024)>
make_ordered_layout(shape, (0, 1, 2, 3)) -> !fly.layout<(8,8,16,2):(1,8,64,1024)>
make_ordered_layout(shape, (3, 2, 1, 0)) -> !fly.layout<(8,8,16,2):(256,32,2,1)>
make_ordered_layout(shape, (1, 0, 3, 2)) -> !fly.layout<(8,8,16,2):(8,1,128,64)>
make_layout(shape, (1, 8, 64, 1024))      -> !fly.layout<(8,8,16,2):(1,8,64,1024)>


- `order=0` 和 `(0, 1, 2, 3)` 结果相同：mode 0 最快，每一维的 stride 是前面各维 shape 的乘积。v0 全用这种写法。
- `order` 里数字越小越快：`(3, 2, 1, 0)` 是最后一维最快；`(1, 0, 3, 2)` 让 mode 1 最快、mode 0 其次，依此类推。
- shape 全是 Python int 时 layout 完全静态，类型里没有 `?`。

### v0 的 A layout

v0 把 `A[M, K]`（行主序，K 连续）看成 5 维：`(8, K//8, 16, BM//16, M//BM)`，order=0。下面取小尺寸 `M=K=64, BM=32`，坐标记作 `(e, kk, r, i, b)`：

$$
\text{offset} = e + 8\,kk + K\,r + 16K\,i + BM \cdot K\,b = \underbrace{(r + 16i + BM\,b)}_{m} \cdot K + \underbrace{(e + 8\,kk)}_{k}
$$

也就是 `A[m, k]`。在 host 上用 `torch.as_strided` 套同一组 shape/stride 就能验证。

In [4]:
M, K, BM = 64, 64, 32
A = torch.randn(M, K, dtype=torch.bfloat16, device="cuda")
A_shape = (8, K // 8, 16, BM // 16, M // BM)
A_stride = (1, 8, K, 16 * K, BM * K)
Av = A.as_strided(A_shape, A_stride)        # 和 FlyDSL layout 相同的 shape:stride


def show_v0_layout(A: fx.Tensor, O: fx.Tensor):
    print("FlyDSL:", fx.make_ordered_layout(A_shape, 0).type)


trace(show_v0_layout, A, O)
print("torch :", A_shape, Av.stride())

e, kk, r, i, b = 3, 5, 7, 1, 1
print("Av[e, kk, r, i, b] =", Av[e, kk, r, i, b].item(), "| A[r+16i+BM*b, e+8kk] =", A[r + 16 * i + BM * b, e + 8 * kk].item())
print("permute 回 [M, K] 与 A 相同:", torch.equal(Av.permute(4, 3, 2, 1, 0).reshape(M, K), A))

FlyDSL: !fly.layout<(8,8,16,2,2):(1,8,64,1024,2048)>
torch : (8, 8, 16, 2, 2) (1, 8, 64, 1024, 2048)
Av[e, kk, r, i, b] = -0.515625 | A[r+16i+BM*b, e+8kk] = -0.515625
permute 回 [M, K] 与 A 相同: True


## 3. `make_view` 与切片

`fx.make_view(iter, layout)` 用一个指针和一个 layout 组成新的 tensor，不搬数据。

切片规则：坐标里写 `None` 的维保留下来，给了值的维被去掉，那个值只用来移动指针。所有维都给值时，`T[coord]` 读出一个元素；`T.load()` 则把整个（子）tensor 读成一个 vector。

In [5]:
def show_view(A: fx.Tensor, O: fx.Tensor):
    tid, bid = fx.thread_idx.x, fx.block_idx.x
    V = fx.make_view(fx.get_iter(A), fx.make_ordered_layout(A_shape, 0))
    print("V                          :", V.type)
    Vb = V[None, None, None, None, bid]
    print("Vb = V[None, None, None, None, bid]:", Vb.type)
    T = Vb[None, tid // 16, tid % 16, 1]
    print("T  = Vb[None, tid//16, tid%16, 1]  :", T.type)
    print("T.load()                   :", T.load().type)
    print("T[3]                       :", type(T[3]).__name__)


trace(show_view, A, O)

V                          : !fly.memref<bf16, global, (8,8,16,2,2):(1,8,64,1024,2048)>
Vb = V[None, None, None, None, bid]: !fly.memref<bf16, global, (8,8,16,2):(1,8,64,1024)>
T  = Vb[None, tid//16, tid%16, 1]  : !fly.memref<bf16, global, 8:1>
T.load()                   : vector<8xbf16>
T[3]                       : BFloat16


- `bid`、`tid` 都是运行时的值，但切完之后的类型仍是静态 layout：类型只由哪些维写了 `None` 决定，坐标值只影响指针。
- 给定 `kk`、`row`、`i` 后剩下 `8:1`，就是 8 个连续 bf16，`.load()` 得到 `vector<8xbf16>`；`T[3]` 只读一个元素，得到标量 `BFloat16`。

在 GPU 上验证：每个 CTA 取 `bid` 块，lane `l` 读 `Vb[:, l//16, l%16, 1]` 写到 `O[bid, l, :]`，和 torch 的 `Av` 对比。

In [6]:
def copy_tile(A: fx.Tensor, O: fx.Tensor):
    tid, bid = fx.thread_idx.x, fx.block_idx.x
    V = fx.make_view(fx.get_iter(A), fx.make_ordered_layout(A_shape, 0))
    Ov = fx.make_view(fx.get_iter(O), fx.make_ordered_layout((8, 64, M // BM), 0))
    Vb = V[None, None, None, None, bid]
    Ov[None, tid, bid].store(Vb[None, tid // 16, tid % 16, 1].load())


O = torch.zeros(M // BM, 64, 8, dtype=torch.bfloat16, device="cuda")
run(copy_tile, A, O, grid=(M // BM, 1, 1))

lane = torch.arange(64)
expect = Av[:, lane // 16, lane % 16, 1, :].permute(2, 1, 0)   # [b, lane, e]
print("GPU 结果与 torch.as_strided 一致:", torch.equal(O, expect))

GPU 结果与 torch.as_strided 一致: True


## 4. 坐标在 ISA 里去了哪里

同一个 `copy_tile` 编译成 gfx942 汇编：

In [7]:
print("\n".join(body(isa(make_launcher(copy_tile, grid=(M // BM, 1, 1)), A, O, stream))))

s_load_dwordx2 s[4:5], s[0:1], 0x0
s_load_dwordx2 s[6:7], s[0:1], 0x18
v_lshrrev_b32_e32 v1, 1, v0
v_lshlrev_b32_e32 v2, 6, v0
s_lshl_b32 s0, s2, 11
v_and_b32_e32 v1, 24, v1
v_and_b32_e32 v2, 0x3c0, v2
v_or3_b32 v2, s0, v2, v1
v_ashrrev_i32_e32 v3, 31, v2
s_waitcnt lgkmcnt(0)
v_lshl_add_u64 v[2:3], v[2:3], 1, s[4:5]
global_load_dwordx4 v[2:5], v[2:3], off offset:2048
s_lshl_b32 s0, s2, 9
v_lshl_or_b32 v0, v0, 3, s0
v_ashrrev_i32_e32 v1, 31, v0
v_lshl_add_u64 v[0:1], v[0:1], 1, s[6:7]
s_waitcnt vmcnt(0)
global_store_dwordx4 v[0:1], v[2:5], off
s_endpgm


对照源码看：

| 源码 | ISA |
|---|---|
| 两个 tensor 的指针 | `s_load_dwordx2` 从 kernarg 读进 SGPR（偏移 `0x0`、`0x18`） |
| A 的 `bid`（一个 wave 内都相同） | 标量指令 `s_lshl_b32 s0, s2, 11`：$bid \cdot BM \cdot K = bid \ll 11$ 个元素，在 SGPR 里算 |
| `tid // 16`、`tid % 16` | 向量指令 `v_lshrrev/v_lshlrev/v_and`，在 VGPR 里算，再用 `v_or3_b32` 和 `bid` 偏移合并 |
| 元素 → 字节 | `v_lshl_add_u64 ..., 1, s[4:5]`：偏移左移 1（bf16 是 2 B）后加到 64 位基址上 |
| 编译期常数 `i = 1` | 立即数 `offset:2048`（$16K \cdot 2\text{B}$），不占指令 |
| `.load()` 8 个连续 bf16 | 一条 `global_load_dwordx4`（16 B） |
| O 的 `bid` 与 `.store()` | `s_lshl_b32 s0, s2, 9`（$bid \cdot 512$ 个元素），一条 `global_store_dwordx4` |

- `make_view` 和切片本身不生成指令；所有 stride 都是常数，乘法都变成了移位，加法变成了或（各项的二进制位不重叠）。
- 元素偏移是 32 位有符号整数：这里先 `v_ashrrev_i32 ... 31` 符号扩展，再加到 64 位基址上。后面第 5、6 节的 kernel 不含 `bid`，偏移只由 `tid` 决定（`known_block_size` 告诉编译器 `tid < 64`），编译器改用 "SGPR 基址 + 32 位 VGPR 偏移" 的形式（如 `global_load_ushort v4, v1, s[2:3]`），省掉了 64 位加法。

## 5. 向量宽度由 layout 决定，不由 `.load()` 决定

把切片换成 stride=8 的那一维（固定 `e`，保留 `kk`），同样是 8 个 bf16：

In [8]:
def strided_load(A: fx.Tensor, O: fx.Tensor):
    tid = fx.thread_idx.x
    V = fx.make_view(fx.get_iter(A), fx.make_ordered_layout(A_shape, 0))
    Ov = fx.make_view(fx.get_iter(O), fx.make_ordered_layout((8, 64), 0))
    T = V[tid % 8, None, tid // 8, 0, 0]
    print("T:", T.type)
    Ov[None, tid].store(T.load())


O = torch.zeros(64, 8, dtype=torch.bfloat16, device="cuda")
print("\n".join(body(isa(make_launcher(strided_load), A, O, stream))))

run(strided_load, A, O)
expect = Av[lane % 8, :, lane // 8, 0, 0]      # [lane, kk]
print("GPU 结果正确:", torch.equal(O, expect))

T: !fly.memref<bf16, global, 8:8>
s_load_dwordx2 s[2:3], s[0:1], 0x0
s_load_dwordx2 s[4:5], s[0:1], 0x18
v_lshl_or_b32 v1, v0, 3, v0
v_and_b32_e32 v1, 0x1c7, v1
v_lshlrev_b32_e32 v1, 1, v1
s_waitcnt lgkmcnt(0)
global_load_ushort v4, v1, s[2:3] offset:32
global_load_ushort v5, v1, s[2:3] offset:48
global_load_ushort v2, v1, s[2:3] offset:64
global_load_ushort v3, v1, s[2:3] offset:96
global_load_ushort v6, v1, s[2:3] offset:112
global_load_ushort v7, v1, s[2:3] offset:80
global_load_ushort v8, v1, s[2:3]
global_load_ushort v9, v1, s[2:3] offset:16
s_mov_b32 s0, 0x5040100
v_lshlrev_b32_e32 v10, 4, v0
s_waitcnt vmcnt(6)
v_perm_b32 v1, v5, v4, s0
s_waitcnt vmcnt(3)
v_perm_b32 v3, v6, v3, s0
s_waitcnt vmcnt(2)
v_perm_b32 v2, v7, v2, s0
s_waitcnt vmcnt(0)
v_perm_b32 v0, v9, v8, s0
global_store_dwordx4 v10, v[0:3], s[4:5]
s_endpgm
T: !fly.memref<bf16, global, 8:8>
GPU 结果正确: True


- 子 tensor 是 `8:8`，元素不连续，`.load()` 被拆成 8 条 2 B 的 `global_load_ushort`，再用 `v_perm_b32` 两两拼成 dword。
- 写出时 `Ov[None, tid]` 是 `8:1`，仍是一条 `global_store_dwordx4`。

所以 `.load()/.store()` 只表示"读/写整个子 tensor"，用几条什么宽度的指令由子 tensor 的 layout 决定。v0 把 K 拆成 `(8, K//8)` 并让 8 那一维 stride=1，就是为了让每个 lane 一次读满 16 B。

## 6. 为什么 v0 不直接用传入的 layout

同样读 `A[tid % 32, tid // 32]`，一个用参数自带的动态 layout，一个用 `make_view` 换成静态 layout：

In [9]:
A = torch.randn(32, 64, dtype=torch.bfloat16, device="cuda")
O = torch.zeros(64, 8, dtype=torch.bfloat16, device="cuda")


def via_arg_layout(A: fx.Tensor, O: fx.Tensor):
    tid = fx.thread_idx.x
    O[tid, 0] = A[tid % 32, tid // 32]


def via_static_layout(A: fx.Tensor, O: fx.Tensor):
    tid = fx.thread_idx.x
    A = fx.make_view(fx.get_iter(A), fx.make_ordered_layout((32, 64), (1, 0)))
    O = fx.make_view(fx.get_iter(O), fx.make_ordered_layout((64, 8), (1, 0)))
    O[tid, 0] = A[tid % 32, tid // 32]


expect = A[lane % 32, lane // 32]
for fn in [via_arg_layout, via_static_layout]:
    O.zero_()
    run(fn, A, O)
    text = isa(make_launcher(fn), A, O, stream)
    print(f"==== {fn.__name__}: GPU 结果正确 = {torch.equal(O[:, 0], expect.to(O.device))}")
    print("\n".join(body(text)))

args = re.findall(r"\.offset:\s+(\d+)\s+\.size:\s+(\d+)\s+\.value_kind:\s+(\w+)", text)
print("==== kernarg (offset, size, kind):", args)

==== via_arg_layout: GPU 结果正确 = True
s_load_dwordx4 s[4:7], s[0:1], 0x10
s_load_dwordx2 s[2:3], s[0:1], 0x0
v_and_b32_e32 v1, 31, v0
v_lshrrev_b32_e32 v6, 4, v0
s_load_dwordx2 s[0:1], s[0:1], 0x28
s_waitcnt lgkmcnt(0)
v_mad_u64_u32 v[2:3], s[8:9], s4, v1, 0
v_mov_b32_e32 v4, v3
v_mad_u64_u32 v[4:5], s[4:5], s5, v1, v[4:5]
v_mov_b32_e32 v3, v4
v_lshl_add_u64 v[2:3], v[2:3], 1, s[2:3]
v_and_b32_e32 v4, 2, v6
v_mov_b32_e32 v5, 0
v_lshl_add_u64 v[2:3], v[2:3], 0, v[4:5]
global_load_ushort v5, v[2:3], off
v_mad_u64_u32 v[2:3], s[2:3], s0, v0, 0
v_mov_b32_e32 v4, v3
s_waitcnt vmcnt(0)
v_mad_u64_u32 v[0:1], s[0:1], s1, v0, v[4:5]
v_mov_b32_e32 v3, v0
v_lshl_add_u64 v[0:1], v[2:3], 1, s[6:7]
global_store_short v[0:1], v5, off
s_endpgm
==== via_static_layout: GPU 结果正确 = True
s_load_dwordx2 s[2:3], s[0:1], 0x0
s_load_dwordx2 s[4:5], s[0:1], 0x18
v_lshrrev_b32_e32 v1, 5, v0
v_lshlrev_b32_e32 v2, 6, v0
s_movk_i32 s0, 0x7c0
v_and_or_b32 v1, v2, s0, v1
v_lshlrev_b32_e32 v1, 1, v1
s_waitcnt lgkmcnt(0

- 动态 layout：要从 kernarg 读出 A、O 的 stride（偏移 `0x10`、`0x28`），stride 是 i64，所以用 64 位乘法 `v_mad_u64_u32` 算地址，读写都用 64 位 VGPR 地址。
- 静态 layout：stride 是常数，乘法变成移位，地址只要一个 32 位 VGPR 偏移加 SGPR 基址。
- 两种写法都没有任何比较/分支：shape 只是记录下来，读写不做越界检查。
- 不管 kernel 用不用，每个 `fx.Tensor` 参数在 kernarg 里都占 24 B：8 B 指针 + 16 B 的 shape/stride。

v0 在 `build_v0(BM, BN, OCCUPANCY, M, N, K)` 里拿到的 M、N、K 都是 Python int，于是丢掉参数的 layout，用 `make_view` 换成全静态的 layout。代价是每个形状都要单独编译一次。

## 7. 回到 v0

用小尺寸 `M=N=128, K=256, BM=BN=64` 编译 v0 本身，看循环体里的访存，并在本机 GPU 上检查结果。

In [10]:
import sys
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path("../kernels/gemm").resolve()))
from v0 import build_v0

M, N, K, BM, BN = 128, 128, 256, 64, 64
A = (torch.randn(M, K, device="cuda") / K ** 0.5).to(torch.bfloat16)
B = (torch.randn(N, K, device="cuda") / K ** 0.5).to(torch.bfloat16)
C = torch.zeros(M, N, dtype=torch.float32, device="cuda")

lines = body(isa(build_v0(BM, BN, 1, M, N, K), A, B, C, stream))
start = lines.index(".LBB0_1:")
end = next(n for n, l in enumerate(lines) if l.startswith("s_cbranch_scc1"))
loop, after = lines[start:end + 1], lines[end + 1:]

ops = lambda part: Counter(l.split()[0] for l in part if l.startswith(("global_", "v_mfma")))
print("循环体:", dict(ops(loop)))
print("循环后:", dict(ops(after)))
print("循环计数:", [l for l in loop if l.startswith(("s_add_u32", "s_cmp"))])
print("A/B 地址计算与 load（循环体前 24 条相关指令）:")
print("  " + "\n  ".join([l for l in loop if l.startswith(("v_or", "v_ashr", "v_lshl_add", "global_load"))][:24]))
print("C 的前 4 条 store:")
print("  " + "\n  ".join([l for l in after if l.startswith("global_store")][:4]))

build_v0(BM, BN, 1, M, N, K)(A, B, C, stream)
torch.cuda.synchronize()
err = (C - A.float() @ B.float().T).abs().max().item()
print(f"v0 GPU 结果最大误差: {err:.2e}")

循环体: {'global_load_dwordx4': 16, 'v_mfma_f32_16x16x16_bf16': 64}
循环后: {'global_store_dword': 64}
循环计数: ['s_add_u32 s12, s12, 2', 's_cmp_lg_u64 s[12:13], 8']
A/B 地址计算与 load（循环体前 24 条相关指令）:
  v_lshl_add_u32 v66, s12, 5, v69
  v_or_b32_e32 v70, s8, v66
  v_ashrrev_i32_e32 v71, 31, v70
  v_or_b32_e32 v75, 0x1000, v66
  v_or_b32_e32 v82, 0x2000, v66
  v_lshl_add_u64 v[70:71], v[70:71], 1, s[4:5]
  v_or_b32_e32 v83, 0x3000, v66
  v_or_b32_e32 v72, s10, v66
  v_or_b32_e32 v74, s8, v75
  v_or_b32_e32 v78, s10, v82
  global_load_dwordx4 v[84:87], v[70:71], off
  v_ashrrev_i32_e32 v73, 31, v72
  v_or_b32_e32 v76, s10, v75
  v_or_b32_e32 v80, s10, v83
  v_ashrrev_i32_e32 v75, 31, v74
  v_ashrrev_i32_e32 v79, 31, v78
  v_lshl_add_u64 v[72:73], v[72:73], 1, s[6:7]
  v_ashrrev_i32_e32 v77, 31, v76
  v_ashrrev_i32_e32 v81, 31, v80
  v_lshl_add_u64 v[70:71], v[74:75], 1, s[4:5]
  v_lshl_add_u64 v[74:75], v[78:79], 1, s[6:7]
  global_load_dwordx4 v[88:91], v[72:73], off
  global_load_dwordx4 v[96:99], 

对照 v0 源码：

- `NI = NJ = 4`，每次迭代 `NI + NJ = 8` 条 load、`NI·NJ·2 = 32` 条 MFMA。循环体里是 16 和 64，循环计数每次加 2：LLVM 把循环展开了 2 次。
- `A[None, kk, row, i].load()` 是 `global_load_dwordx4`，和第 5 节结论一致。
- `v_lshl_add_u32 v66, s12, 5, v69`：循环变量 `k` 乘 32（$8 \cdot kk = 32k + 8\,kg$）加上 lane 自己的部分。不同 `i` 的偏移是常数 `0x1000, 0x2000, 0x3000`（$16K$ 个元素），用 `v_or` 合并；`s8/s10` 是在 SGPR 里算好的 A/B 的 `bid` 偏移。
- 和第 4 节一样，含 `bid` 的偏移按 32 位有符号数算，再符号扩展（`v_ashrrev_i32 ... 31`）加到 64 位基址上。
- `C[row, j, kg*4 + r, i] = out[r]` 是标量写：每个 lane 的 4 个结果在 C 的 4 个不同行（相隔 N 个元素），只能写 4 次 `global_store_dword`，共 `NI·NJ·4 = 64` 条。同一行的 16 个 lane 写连续的 64 B。
- 结果和 `A.float() @ B.float().T` 的最大误差在 1e-7 量级。

## 总结

| v0 写法 | trace 时 | ISA |
|---|---|---|
| `A: fx.Tensor` 参数 | `!fly.memref`：指针 + 动态 layout `(?,?):(?,1)` | kernarg 里占 24 B；v0 只读其中的指针 |
| （对照）直接用参数的动态 layout | 同上 | 从 kernarg 读 stride，64 位乘法算地址；不做越界检查 |
| `fx.get_iter(A)` | `!fly.ptr<bf16, global>` | 基址在 SGPR |
| `fx.make_ordered_layout(shape, 0)` | 静态 layout，mode 0 最快，stride 自动算 | 不生成指令，stride 变成常数，乘法变移位 |
| `fx.make_view(iter, layout)` | 新的 memref | 不生成指令 |
| `A[None, ..., bid_m]` | 去掉给定值的维，类型仍是静态 layout | `bid` 偏移由标量指令在 SGPR 里算；偏移是 32 位有符号数 |
| `A[None, kk, row, i].load()` | `vector<8xbf16>` | 一条 `global_load_dwordx4`；常数坐标进立即数或常量 |
| stride≠1 的子 tensor `.load()` | 同样是 vector | 拆成逐元素 `global_load_ushort` + `v_perm_b32` |
| `C[...] = x` | 单个元素 | `global_store_dword` |